# enasmblers models 

In [2]:
! pip install category_encoders lightgbm catboost xgboost

In [1]:
import pandas as pd 
import numpy as np 
import sklearn as sk
import lightgbm as lgb 
import catboost as cb 
import xgboost as xgb 
import category_encoders as ce 
from collections import Counter
from scipy.special import logit 
from sklearn.tree import DecisionTreeClassifier , DecisionTreeRegressor
from sklearn.metrics import roc_auc_score , r2_score , classification_report
from sklearn.ensemble import RandomForestClassifier , GradientBoostingClassifier , ExtraTreesClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

### upload the data

In [2]:
train_data = pd.read_csv("../datasets/training.csv")

In [3]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   RefId                              72983 non-null  int64  
 1   IsBadBuy                           72983 non-null  int64  
 2   PurchDate                          72983 non-null  object 
 3   Auction                            72983 non-null  object 
 4   VehYear                            72983 non-null  int64  
 5   VehicleAge                         72983 non-null  int64  
 6   Make                               72983 non-null  object 
 7   Model                              72983 non-null  object 
 8   Trim                               70623 non-null  object 
 9   SubModel                           72975 non-null  object 
 10  Color                              72975 non-null  object 
 11  Transmission                       72974 non-null  obj

In [4]:
train_data.isnull().sum()

RefId                                    0
IsBadBuy                                 0
PurchDate                                0
Auction                                  0
VehYear                                  0
VehicleAge                               0
Make                                     0
Model                                    0
Trim                                  2360
SubModel                                 8
Color                                    8
Transmission                             9
WheelTypeID                           3169
WheelType                             3174
VehOdo                                   0
Nationality                              5
Size                                     5
TopThreeAmericanName                     5
MMRAcquisitionAuctionAveragePrice       18
MMRAcquisitionAuctionCleanPrice         18
MMRAcquisitionRetailAveragePrice        18
MMRAcquisitonRetailCleanPrice           18
MMRCurrentAuctionAveragePrice          315
MMRCurrentA

### check out the missing data by 50% of the data is missing and replace 

In [19]:
train_data

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,12/7/2009,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,4,0,12/7/2009,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,...,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,5,0,12/7/2009,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,...,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72978,73010,1,12/2/2009,ADESA,2001,8,MERCURY,SABLE,GS,4D SEDAN GS,...,4836.0,5937.0,NaN,NaN,18111,30212,GA,4200.0,0,993
72979,73011,0,12/2/2009,ADESA,2007,2,CHEVROLET,MALIBU 4C,LS,4D SEDAN LS,...,10151.0,11652.0,NaN,NaN,18881,30212,GA,6200.0,0,1038
72980,73012,0,12/2/2009,ADESA,2005,4,JEEP,GRAND CHEROKEE 2WD V,Lar,4D WAGON LAREDO,...,11831.0,14402.0,NaN,NaN,18111,30212,GA,8200.0,0,1893
72981,73013,0,12/2/2009,ADESA,2006,3,CHEVROLET,IMPALA,LS,4D SEDAN LS,...,10099.0,11228.0,NaN,NaN,18881,30212,GA,7000.0,0,1974


In [5]:
def missing(train_data:pd.DataFrame):
    max_size  = len(train_data) * 0.5
    col_del = []
    for col in train_data.columns:
     if train_data[col].isna().sum() >= max_size :
        # more than 50% missed lets delete it 
        col_del.append(col)
        train_data = train_data.dropna(col_del , axis=1)
     category = train_data.select_dtypes(include='object').columns.tolist()
     numbers = train_data.select_dtypes(include=np.number).columns.tolist()
     for col in category:
        if train_data[col].isna().sum() != 0 :
            train_data[col] = train_data[col].fillna(train_data[col].mode()[0])
     for col in numbers :
        if train_data[col].isna().sum() != 0:
            train_data[col] = train_data[col].fillna(train_data[col].mean())

     return train_data
   

In [6]:
train_data = missing(train_data=train_data)
train_data.isnull().sum()

RefId                                0
IsBadBuy                             0
PurchDate                            0
Auction                              0
VehYear                              0
VehicleAge                           0
Make                                 0
Model                                0
Trim                                 0
SubModel                             0
Color                                0
Transmission                         0
WheelTypeID                          0
WheelType                            0
VehOdo                               0
Nationality                          0
Size                                 0
TopThreeAmericanName                 0
MMRAcquisitionAuctionAveragePrice    0
MMRAcquisitionAuctionCleanPrice      0
MMRAcquisitionRetailAveragePrice     0
MMRAcquisitonRetailCleanPrice        0
MMRCurrentAuctionAveragePrice        0
MMRCurrentAuctionCleanPrice          0
MMRCurrentRetailAveragePrice         0
MMRCurrentRetailCleanPric

In [7]:
train_data.nunique()


RefId                                72983
IsBadBuy                                 2
PurchDate                              517
Auction                                  3
VehYear                                 10
VehicleAge                              10
Make                                    33
Model                                 1063
Trim                                   134
SubModel                               863
Color                                   16
Transmission                             3
WheelTypeID                              5
WheelType                                3
VehOdo                               39947
Nationality                              4
Size                                    12
TopThreeAmericanName                     4
MMRAcquisitionAuctionAveragePrice    10343
MMRAcquisitionAuctionCleanPrice      11380
MMRAcquisitionRetailAveragePrice     12726
MMRAcquisitonRetailCleanPrice        13457
MMRCurrentAuctionAveragePrice        10316
MMRCurrentA

#### we use quantile beens to we autmatically mixed up vehdo and ferid and drop then for better result 

In [8]:
train_data["VehOdo_Binned"] = pd.qcut(x=train_data["VehOdo"],
                                     q=[0, 0.25, 0.5, 0.75, 1.0],
                                     labels=[0, 1, 2 ,3])
train_data = train_data.drop(["VehOdo", "RefId"], axis=1)
     

Use the "PurchDate" field to split, test must be later in time than validation, same goes for validation and train: train.PurchDate < valid.PurchDate < test.PurchDate.
Use the first 33% of the data for the training, the last 33% of the data for the test, and the middle 33% for the validation set. Don't use the test dataset until the end!

In [11]:
def split_by_time(train_df:pd.DataFrame):
    total = len(train_df)
    train_end_index =  int(total * 0.33)
    valid_end_index = int(total * 0.66)

    train_df['PurchDate'] = pd.to_datetime(train_data['PurchDate'], errors='coerce')
    train_df = train_data.sort_values(by='PurchDate')
    train_df['PurchDate'] = train_df['PurchDate'].astype(int)

    x = train_df.drop(['IsBadBuy'] , axis=1)
    y = train_df['IsBadBuy']

    while x.loc[train_end_index]['PurchDate'] == x.loc[valid_end_index]['PurchDate']:
        valid_end_index += 1
        print(valid_end_index) 
    x_train = x.iloc[:train_end_index]
    x_valid = x.iloc[train_end_index:valid_end_index]
    x_test = x.iloc[valid_end_index:]

    y_train = y.iloc[:train_end_index]
    y_valid = y.iloc[train_end_index:valid_end_index]
    y_test = y.iloc[valid_end_index:]
    return x_train, x_valid, x_test, y_train, y_valid, y_test


In [12]:
X_train, X_valid, X_test, Y_train, Y_valid, Y_test = split_by_time(train_data)
len(X_train), len(X_valid), len(X_test)

(24084, 24084, 24815)

In [13]:
print(max(X_train["PurchDate"]))
print(max(X_valid["PurchDate"]))
print(max(X_test["PurchDate"]))

1252627200000000000
1273536000000000000
1293667200000000000


Use LabelEncoder or OneHotEncoder from sklearn to preprocess categorical variables. Be careful with data leakage (fit Encoder to training and apply to validation & test).

In [14]:
def category_encoded(train_data , x_train , x_valid , x_test):
    cat_Col = train_data.select_dtypes(include='object').columns.tolist()
    count_encoder = ce.CountEncoder(cols=cat_Col)
    train_encoded = count_encoder.fit_transform(x_train)
    valid_encoded = count_encoder.transform(x_valid)
    test_encoded = count_encoder.transform(x_test)
    return train_encoded , valid_encoded , test_encoded

In [15]:
train_encoded, valid_encoded, test_encoded = category_encoded(train_data, X_train, X_valid, X_test)
train_encoded.sample(5)

,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,...,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost,VehOdo_Binned
1286,1239062400000000000,4953,2006,3,2726,258,5193,279,4162,23416,...,11153.0,24082,24084,20392,38118,597,6210.0,0,1630,3
4896,1242950400000000000,13960,2005,4,4747,326,3406,127,4162,23416,...,7007.0,24082,24084,99750,85040,2436,5200.0,0,1623,2
53511,1252368000000000000,5171,2008,1,4747,274,3406,4846,4162,23416,...,10571.0,24082,24084,99761,74135,781,8100.0,0,834,0
26378,1234915200000000000,13960,2006,3,691,227,596,4846,4965,23416,...,6359.0,24082,24084,19619,32824,3373,4900.0,0,482,0
55755,1241136000000000000,5171,2008,1,4747,274,3406,4846,3398,23416,...,10268.0,24082,24084,99761,33073,3373,8200.0,0,834,0


In [16]:
train_encoded.nunique()


PurchDate                             180
Auction                                 3
VehYear                                 8
VehicleAge                              8
Make                                   29
Model                                 128
Trim                                   69
SubModel                              115
Color                                  16
Transmission                            2
WheelTypeID                             4
WheelType                               3
Nationality                             4
Size                                   12
TopThreeAmericanName                    4
MMRAcquisitionAuctionAveragePrice    5983
MMRAcquisitionAuctionCleanPrice      6274
MMRAcquisitionRetailAveragePrice     5983
MMRAcquisitonRetailCleanPrice        6272
MMRCurrentAuctionAveragePrice        6739
MMRCurrentAuctionCleanPrice          7039
MMRCurrentRetailAveragePrice         6904
MMRCurrentRetailCleanPrice           7224
PRIMEUNIT                         

###  Create a Python class for Decision Tree Classifier and Decision Tree Regressor (MSE loss).
* It should support fit,  predict_proba  , and predict methods.
Also, the maximum depth (max_depth) must be a parameter of your class.
Use the Gini impurity criterion as a criterion for choosing the split.
Create a separate class for Node. It should be able to hold data (sample features and targets), compute Gini impurity, and have pointers to children (left and right nodes).

* For the Regressor, use standard deviation instead of Gini impurity.
Implement a function that finds the best possible split in the current node.
Combine the previous steps into your working Decision Tree Classifier.
Implement an Extra Randomized Tree by designing another function to find the best split.

In [17]:
class MyNode:
  def __init__(self, X, Y, feature=None, threshold=None, left=None, right=None,*,value=None, probs=None):
    self.X = X
    self.Y = Y
    self.feature = feature 
    self.threshold  = threshold 
    self.left = left 
    self.right = right 
    self.value = value 
    self.class_probs = probs 

  def is_leaf_node(self):
    return self.value is not None

  def compute_gini(self):
    hist = np.bincount(self.Y)
    ps = hist / len(self.Y)
    return 1 - np.sum(ps ** 2)

  def compute_mse(self):
    n = len(self.Y)
    Y_mean = np.mean(self.Y)
    return np.sum((self.Y - Y_mean)**2)/n

In [19]:
class MyDecisionTreeClassifier:
  def __init__(self, min_samples_split=2, min_samples_leaf=1, max_depth=10, n_features=None, criterion='gini', random_state=21):
    self.min_samples_split = min_samples_split
    self.min_samples_leaf = min_samples_leaf
    self.max_depth = max_depth
    self.n_features = n_features
    self.criterion = criterion
    self.random_state = random_state
    self.root=None
    self.classes_=None

  def fit(self, X, Y):
    X = self._to_numpy(X)
    Y = self._to_numpy(Y)

    np.random.seed(self.random_state)
    self.classes_ = np.unique(Y)
    self.n_features = X.shape[1] if not self.n_features else min(X.shape[1], self.n_features)
    self.root = self._build_tree(X, Y)
    self._prune_tree(self.root)

  def predict(self, X):
    if self.root is None:
      raise ValueError("fit()!")
    X = self._to_numpy(X)
    return np.array([self._traverse_tree(x, self.root) for x in X])

  def predict_proba(self, X):
    if self.root is None:
      raise ValueError(" fit()!")
    X = self._to_numpy(X)
    probas = [self._get_class_probs(x, self.root) for x in X]
    return np.array(probas)

  def _to_numpy(self, data):
    if isinstance(data, (pd.DataFrame, pd.Series)):
        return data.values
    elif isinstance(data, np.ndarray):
        return data
    else:
        return np.array(data)

  def _get_class_probs(self, x, node):
    if node.is_leaf_node():
        return node.class_probs

    if x[node.feature] <= node.threshold:
        return self._get_class_probs(x, node.left)
    return self._get_class_probs(x, node.right)

  def _traverse_tree(self, x, node):
    if node.is_leaf_node():
      return node.value

    if x[node.feature] <= node.threshold:
      return self._traverse_tree(x, node.left)
    return self._traverse_tree(x, node.right)

  def _build_tree(self, X, Y, depth=0):
    n_samples, n_feats = X.shape
    n_labels = len(np.unique(Y))

    # stopping criteria - base case
    if (depth >= self.max_depth or n_labels == 1
        or n_samples < self.min_samples_split
        or n_samples < 2 *self.min_samples_leaf):
      class_probs = self._compute_probs(Y)
      leaf_value = self._find_common_label(Y)
      return MyNode(X, Y, value=leaf_value, probs=class_probs)

    # recursive case
    feat_idx = np.random.choice(n_feats, self.n_features, replace=False)
    # find best split
    best_feature, best_threshold = self._find_best_split(X, Y, feat_idx)
    if best_feature is None:
      class_probs = self._compute_probs(Y)
      leaf_value = self._find_common_label(Y)
      return MyNode(X, Y, value=leaf_value, probs=class_probs)

    # create child nodes
    left_idxs, right_idxs = self._split(X[:, best_feature], best_threshold)
    left = self._build_tree(X[left_idxs, :], Y[left_idxs], depth+1)
    right = self._build_tree(X[right_idxs, :], Y[right_idxs], depth+1)
    return MyNode(X, Y, best_feature, best_threshold, left, right)

  def _find_common_label(self, Y):
    classes = Counter(Y)
    value = classes.most_common(1)[0][0] #[(obj, cnt)]
    return value

  def _compute_probs(self, Y):
    cnt = Counter(Y)
    probs = np.zeros(len(self.classes_))
    for i, cls in enumerate(self.classes_):
      probs[i] = cnt.get(cls, 0) / len(Y)
    return probs


  def _find_best_split(self, X, Y, feat_idxs):
    split_idx, split_threshold = None, None

    if self.criterion == 'gain':
      best_value = -1
      compute_func = self._compute_gain

    elif self.criterion == 'gini':
      best_value = 0
      compute_func = self._compute_delta_gini
    else:
        return None, None

    for feat_idx in feat_idxs:
        X_column = X[:, feat_idx]

        thresholds = np.unique(X_column)

        if len(thresholds) > 100:
          percentiles = np.linspace(0, 100, 100)
          thresholds = np.percentile(X_column, percentiles)

        for thr in thresholds:
            left_idxs, right_idxs = self._split(X_column, thr)
            # проверка на min_samples_leaf
            if len(left_idxs) < self.min_samples_leaf or len(right_idxs) < self.min_samples_leaf:
                continue

            current_value = compute_func(Y, X_column, thr)
            if current_value > best_value:
                best_value = current_value
                split_idx = feat_idx
                split_threshold = thr
    return split_idx, split_threshold

  def _split(self, X_column, split_threshold):
    # [[][][]] > []
    left_mask = X_column <= split_threshold
    right_mask = ~left_mask

    left_idxs = np.where(left_mask)[0]
    right_idxs = np.where(right_mask)[0]

    #left_idxs = np.argwhere(X_column<=split_threshold).flatten()
    #right_idxs =  np.argwhere(X_column>split_threshold).flatten()
    return left_idxs, right_idxs

  def _prune_tree(self, node):
        node.X = None
        if node.left:
            self._prune_tree(node.left)
        if node.right:
            self._prune_tree(node.right)

  def _compute_delta_gini(self, Y, X_column, threshold):
    left_mask = X_column <= threshold
    right_mask = ~left_mask

    n_left = np.sum(left_mask)
    n_right = np.sum(right_mask )

    if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
        return 0

    n = len(Y)
    p_parent = np.mean(Y)

    # дочерн узлы
    p_left = np.mean(Y[left_mask]) if n_left > 0 else 0
    p_right = np.mean(Y[right_mask]) if n_right > 0 else 0

    gini_parent = 2 * p_parent * (1 - p_parent)
    gini_left = 2 * p_left * (1 - p_left) if n_left > 0 else 0
    gini_right = 2 * p_right * (1 - p_right) if n_right > 0 else 0
    delta_gini = gini_parent - (n_left/n) * gini_left - (n_right/n) * gini_right

    return delta_gini


  def _compute_gain(self, Y, X_column, threshold):
    # parent enthropy
    parent_enthropy = self._enthropy(Y)

    # create children
    left_idxs, right_idxs = self._split(X_column, threshold)
    if len(left_idxs) == 0 or len(right_idxs) == 0:
      return 0

    # calc weighted entr of children
    n = len(Y)
    n_left, n_right = len(left_idxs), len(right_idxs)
    ent_left, ent_right = self._enthropy(Y[left_idxs]), self._enthropy(Y[right_idxs])
    child_enthropy = (n_left/n) * ent_left + (n_right/n) * ent_right

    # IG
    inf_gain = parent_enthropy - child_enthropy
    return inf_gain

  def _enthropy(self, Y):
    hist = np.bincount(Y)
    ps = hist / len(Y)
    return -np.sum([p*np.log(p) for p in ps if p>0])


In [20]:
def gini_21(Y_true:np.array, Y_predicted:np.array):
  gini_score = 2 * roc_auc_score(Y_true, Y_predicted) - 1
  return gini_score

##### With your DecisionTree module, you must obtain a Gini score of at least 0.1 on the validation dataset.
yes 

##### . Use sklearn's DecisionTreeClassifier and check its performance on the validation dataset. Is it better than your module? If so, why?

not so much but yes and due to predmeterts and accuracy

In [21]:
my_classifier = MyDecisionTreeClassifier(max_depth=15, min_samples_split=100, n_features=10)
my_classifier.fit(train_encoded, Y_train)

In [22]:
Y_pred_my_proba = my_classifier.predict_proba(valid_encoded)[:, 1]
my_gini = gini_21(Y_valid, Y_pred_my_proba)
print(f"My gini_score: {my_gini:.4f}")

My gini_score: 0.3140


In [23]:


skl_tree = DecisionTreeClassifier(max_depth=15, min_samples_split=100, max_features=10, random_state=21)
skl_tree.fit(train_encoded, Y_train)

DecisionTreeClassifier(max_depth=15, max_features=10, min_samples_split=100,
                       random_state=21)

In [24]:
Y_pred_skl_proba = skl_tree.predict_proba(valid_encoded)[:, 1]
skl_gini = gini_21(Y_valid, Y_pred_skl_proba)
print(f"Sklearn gini_score: {skl_gini:.4f}")
#0.3000

Sklearn gini_score: 0.3564


* The Gini-score turns out to be better than in the sklearn implementation because my algorithm uses random feature subsampling, while the sklearn model uses an optimized deterministic set of thresholds (based on quantiles).


### regressor

In [25]:
class MyDecisionTreeRegressor:
  def __init__(self, min_samples_split=2, min_samples_leaf=1, max_depth=10,n_features=None, criterion='squared_error', random_state=21):
    self.min_samples_split = min_samples_split
    self.min_samples_leaf = min_samples_leaf
    self.max_depth = max_depth
    self.n_features = n_features
    self.criterion = criterion
    self.random_state = random_state
    self.root=None
    self.initial_variance_=None

  def fit(self, X, Y):
    X = self._to_numpy(X)
    Y = self._to_numpy(Y)
    self.initial_variance_ = np.var(Y)
    np.random.seed(self.random_state)
    self.n_features = X.shape[1] if not self.n_features else min(X.shape[1], self.n_features)
    self.root = self._build_tree(X, Y)
    return self

  def predict(self, X):
    if self.root is None:
      raise ValueError("Дерево еще не обучено методом fit()!")
    X = self._to_numpy(X)
    return np.array([self._traverse_tree(x, self.root) for x in X])

  def _traverse_tree(self, x, node):
    if node.is_leaf_node():
      return node.value

    if x[node.feature] <= node.threshold:
      return self._traverse_tree(x, node.left)
    return self._traverse_tree(x, node.right)

  def _build_tree(self, X, Y, depth=0):
    n_samples, n_feats = X.shape
    min_variance_ratio = 0.01

    # base case
    if (depth >= self.max_depth
        or np.var(Y) < min_variance_ratio * self.initial_variance_
        or n_samples < self.min_samples_split
        or n_samples < 2 *self.min_samples_leaf):
      leaf_value = np.mean(Y)
      return MyNode(X, Y, value=leaf_value)

    # recursive case
    feat_idx = np.random.choice(n_feats, self.n_features, replace=False)
    best_feature, best_threshold = self._find_best_split(X, Y, feat_idx)
    if best_feature is None:
      leaf_value = np.mean(Y)
      return MyNode(X, Y, value=leaf_value)

    # create child nodes
    left_idxs, right_idxs = self._split(X[:, best_feature], best_threshold)
    left = self._build_tree(X[left_idxs, :], Y[left_idxs], depth+1)
    right = self._build_tree(X[right_idxs, :], Y[right_idxs], depth+1)
    return MyNode(X, Y, best_feature, best_threshold, left, right)

  def _find_best_split(self, X, Y, feat_idxs):
    split_idx, split_threshold = None, None
    if self.criterion == 'squared_error':
      # цель - уменьшение mse
      best_delta_mse = 0
      for feat_idx in feat_idxs:
        X_column = X[:, feat_idx]

        thresholds = np.unique(X_column)
        if len(thresholds) > 100:
          percentiles = np.linspace(0, 100, 100)
          thresholds = np.percentile(X_column, percentiles)

        for thr in thresholds:
          left_idxs, right_idxs = self._split(X_column, thr)

          if len(left_idxs) < self.min_samples_leaf or len(right_idxs) < self.min_samples_leaf:
              continue

          delta_mse = self._compute_delta_mse(Y, X_column, thr)
          if  delta_mse > best_delta_mse:
            best_delta_mse = delta_mse
            split_idx = feat_idx
            split_threshold = thr
    return split_idx, split_threshold

  def _split(self, X_column, split_threshold):
    left_mask = X_column <= split_threshold
    right_mask = ~left_mask

    left_idxs = np.where(left_mask)[0]
    right_idxs = np.where(right_mask)[0]
    return left_idxs, right_idxs

  def _compute_delta_mse(self,Y, X_column, threshold):
    left_mask = X_column <= threshold
    right_mask = ~left_mask

    n_left = np.sum(left_mask)
    n_right = np.sum(right_mask)
    n = len(Y)

    if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
        return 0.0

    parent_mse = np.var(Y, ddof=0)
    Y_left = Y[left_mask]
    Y_right = Y[right_mask]
    ss_left = np.sum((Y_left - np.mean(Y_left))**2) if n_left > 0 else 0.0
    ss_right = np.sum((Y_right - np.mean(Y_right))**2) if n_right > 0 else 0.0
    weighted_children_mse = (ss_left + ss_right) / n

    delta_mse = parent_mse - weighted_children_mse
    return max(delta_mse, 0.0)

  def _to_numpy(self, data):
    if isinstance(data, (pd.DataFrame, pd.Series)):
        return data.values
    elif isinstance(data, np.ndarray):
        return data
    else:
        return np.array(data)

In [26]:
my_regressor = MyDecisionTreeRegressor(max_depth=15, min_samples_split=100, n_features=10)
my_regressor.fit(train_encoded, Y_train)

In [37]:
Y_pred_my = my_regressor.predict(valid_encoded)
my_r2 = r2_score(Y_valid, Y_pred_my)
print(f"My r2_score on valid: {my_r2:.4f}")
# -0.0994

My r2_score on valid: -0.1150


In [38]:
Y_pred_my = my_regressor.predict(train_encoded)
my_r2 = r2_score(Y_train, Y_pred_my)
print(f"My r2_score on train: {my_r2:.4f}")
# 0.2951

My r2_score on train: 0.2884


In [39]:
skl_rtree = DecisionTreeRegressor(max_depth=15, min_samples_split=100, max_features=10, random_state=21)
skl_rtree.fit(train_encoded, Y_train)

DecisionTreeRegressor(max_depth=15, max_features=10, min_samples_split=100,
                      random_state=21)

In [40]:
Y_pred_skl = skl_rtree.predict(valid_encoded)
skl_r2 = r2_score(Y_valid, Y_pred_skl)
print(f"Sklearn r2_score on valid: {skl_r2:.4f}")
# -0.3592

Sklearn r2_score on valid: -0.1354


In [42]:
Y_pred_skl = skl_rtree.predict(train_encoded)
skl_r2 = r2_score(Y_train, Y_pred_skl)
print(f"Sklearn r2_score on train: {skl_r2:.4f}")
# 0.2686
     

Sklearn r2_score on train: 0.2721


* However, the original also overfits, strongly adapts to the training dataset, and ends up performing significantly worse than simply predicting the mean value.
As a result, my tree turns out to be more strongly regularized due to the additional stopping criterion (stopping when the variance in a node is less than 1% of the initial value).

### extra tree randomize ( classifier)

In [27]:
class MyExtraRandomizedTree(MyDecisionTreeClassifier):
  def __init__(self, min_samples_split=2, min_samples_leaf=1, max_depth=10, n_features=None, criterion='gini', random_state=21, n_random_thresholds=5):
    super().__init__(min_samples_split=min_samples_split,
                         min_samples_leaf=min_samples_leaf,
                         max_depth=max_depth,
                         n_features=n_features,
                         criterion=criterion,
                         random_state=random_state)
    self.n_random_thresholds = n_random_thresholds

  def _find_best_split(self, X, Y, feat_idxs):
    split_idx, split_threshold = None, None

    if self.criterion == 'gain':
        best_value = -1
        compute_func = self._compute_gain
    elif self.criterion == 'gini':
        best_value = 0
        compute_func = self._compute_delta_gini
    else:
        return None, None

    for feat_idx in feat_idxs:
        X_column = X[:, feat_idx]
        unique_values = np.unique(X_column)
        n_possible = min(len(unique_values), self.n_random_thresholds)
        thresholds = np.random.choice(unique_values, size=n_possible, replace=False)
        for thr in thresholds:
            left_idxs, right_idxs = self._split(X_column, thr)

            if len(left_idxs) < self.min_samples_leaf or len(right_idxs) < self.min_samples_leaf:
                continue

            current_value = compute_func(Y, X_column, thr)

            if current_value > best_value:
                best_value = current_value
                split_idx = feat_idx
                split_threshold = thr

    return split_idx, split_threshold
     

In [28]:
my_classifier = MyExtraRandomizedTree(max_depth=15, min_samples_split=100, n_features=10)
my_classifier.fit(train_encoded, Y_train)

In [30]:
Y_pred_my_proba = my_classifier.predict_proba(valid_encoded)[:,1]
my_gini = gini_21(Y_valid, Y_pred_my_proba)
print(f"My gini_score: {my_gini:.4f}")
#0.3620

My gini_score: 0.2967


### Implement the RandomForestClassifier and check its performance.

In [32]:
class MyRandomForestClassifier:
  def __init__(self, n_estimators=100, criterion='gini', max_depth=15,min_samples_split=100,min_samples_leaf=1,n_features=10, random_state=21, max_samples=None):
    self.n_estimators = n_estimators
    self.criterion = criterion
    self.max_depth = max_depth
    self.min_samples_split = min_samples_split
    self.min_samples_leaf = min_samples_leaf
    self.n_features = n_features
    self.random_state = random_state
    self.max_samples = max_samples # абс число, не доля
    self.trees = []

  def fit(self, X, Y):
    X = self._to_numpy(X)
    Y = self._to_numpy(Y)
    self.classes_ = np.unique(Y)
    self.n_classes_ = len(self.classes_)
    self.trees = []
    rng = np.random.default_rng(self.random_state)

    for i in range(self.n_estimators):
      tree_seed = rng.integers(0, 2**31 - 1)
      X_sample, Y_sample = self._bootstrap_sample(X, Y, rng)
      tree = MyDecisionTreeClassifier(self.min_samples_split,
                                      self.min_samples_leaf,
                                      self.max_depth,
                                      self.n_features,
                                      self.criterion,
                                      random_state=tree_seed)
      tree.fit(X_sample, Y_sample)
      self.trees.append(tree)

  def predict(self, X):
    if not self.trees:
      raise ValueError("Лес еще не обучен методом fit()!")
    X = self._to_numpy(X)
    all_preds = np.array([tree.predict(X) for tree in self.trees])

    n_samples = X.shape[0]
    final_predictions = np.zeros(n_samples, dtype=self.classes_.dtype)

    for sample_idx in range(n_samples):
        # голоса всех деревьев для одного образца
        votes = all_preds[:, sample_idx]
        # голоса в индексы self.classes_
        vote_indices = np.searchsorted(self.classes_, votes)
        # голоса с учетом длины n_classes_
        counts = np.bincount(vote_indices, minlength=self.n_classes_)
        final_predictions[sample_idx] = self.classes_[np.argmax(counts)]
    return final_predictions

  def predict_proba(self, X):
    if not self.trees:
      raise ValueError("Лес еще не обучен методом fit()!")
    X = self._to_numpy(X)
    n_samples = X.shape[0]

    # вер-ти от ВСЕХ трис
    all_proba = np.zeros((self.n_estimators, n_samples, self.n_classes_))
    for i, tree in enumerate(self.trees):
        # от 1
        tree_proba = tree.predict_proba(X)  # (n_samples, n_classes)
        for j, cls in enumerate(tree.classes_):
            # индекс класса в общем списке классов леса
            class_index = np.where(self.classes_ == cls)[0]
            if len(class_index) > 0:
                all_proba[i, :, class_index[0]] = tree_proba[:, j]
    avg_proba = np.mean(all_proba, axis=0)
    return avg_proba

  def _bootstrap_sample(self, x, y, rng):
    n_samples = x.shape[0]
    sample_size = n_samples if not self.max_samples else min(self.max_samples, n_samples)
    idxs = rng.choice(n_samples, sample_size, replace=True)
    return x[idxs], y[idxs]

  def _to_numpy(self, data):
    if isinstance(data, (pd.DataFrame, pd.Series)):
        return data.values
    elif isinstance(data, np.ndarray):
        return data
    else:
        return np.array(data)

In [34]:
my_classifier = MyRandomForestClassifier(n_estimators=50,max_depth=15, min_samples_split=10, n_features=10, max_samples=100)
my_classifier.fit(train_encoded, Y_train)

In [35]:
Y_pred_my = my_classifier.predict_proba(valid_encoded)[:,1]
my_gini = gini_21(Y_valid, Y_pred_my)
print(f"My gini_score: {my_gini:.4f}")


My gini_score: 0.1562


In [36]:
skl_forest = RandomForestClassifier(n_estimators=50,max_depth=15, min_samples_split=10, max_features=10, random_state=21, max_samples=100)
skl_forest.fit(train_encoded, Y_train)

RandomForestClassifier(max_depth=15, max_features=10, max_samples=100,
                       min_samples_split=10, n_estimators=50, random_state=21)

In [37]:
Y_pred_skl = skl_forest.predict_proba(valid_encoded)[:,1]
skl_gini = gini_21(Y_valid, Y_pred_skl)
print(f"Skl gini_score: {skl_gini:.4f}")


Skl gini_score: 0.2959


* With equal parameters, my implementation and sklearn's are comparable.

### Use your DecisionTree design class for GBDT classifier.

This class must have max_depth, number_of_trees and max_features attributes.
You must compute the gradient of the binary cross-entropy loss function and
implement incremental learning: train the next tree using the results of the previous trees.

In [38]:
class MyGradientBoosting:
  def __init__(self, max_depth=15, number_of_trees=5, max_features=10, base_learner=MyDecisionTreeRegressor, random_state=21, learning_rate=0.001):
    self.max_depth = max_depth
    self.number_of_trees = number_of_trees
    self.max_features = max_features
    self.base_learner = base_learner
    self.random_state = random_state
    self.learning_rate = learning_rate

  def fit(self, X, Y):
    X = self._to_numpy(X)
    Y = self._to_numpy(Y)
    self.classes_ = np.unique(Y)
    self.n_classes_ = len(self.classes_)
    self.trees = []
    y = (Y == self.classes_[1]).astype(float)

    pos = np.mean(y)
    F0 = np.full_like(y, logit(pos), dtype=float)

    self.X_train = X
    self.Y_train = y
    for i in range(self.number_of_trees):
      p = 1.0 / (1.0 + np.exp(-F0))
      res = y - p
      tree = self.base_learner(max_depth=self.max_depth, n_features= self.max_features, random_state = self.random_state+i)
      tree.fit(X, res)
      self.trees.append(tree)

      tree_pred = tree.predict(X)
      F0 += self.learning_rate * tree_pred
      p_curr = 1.0 / (1.0 + np.exp(-F0))
      curr_loss = self._compute_cross_enthropy(y, p_curr)
      print(f"Дерево {i}, Loss {curr_loss:.4f}")


  def predict(self, X):
    threshold = 0.5
    proba = self.predict_proba(X)
    result = (proba[:, 1] > threshold).astype(int)
    return result

  def predict_proba(self, X):
    X = self._to_numpy(X)
    pos = np.mean(self.Y_train)
    F = np.full(X.shape[0], logit(pos), dtype=float)

    for tree in self.trees:
      F += self.learning_rate * tree.predict(X)
    proba = 1.0 / (1.0 + np.exp(-F))
    result = np.column_stack([1-proba, proba])
    return result

  def _compute_cross_enthropy(self, y_true, pred):
    eps = 1e-15
    pred = np.clip(pred, eps, 1-eps)
    ce = - np.mean(y_true * np.log(pred) + (1 - y_true) * np.log(1 - pred))
    return ce

  def _to_numpy(self, data):
    if isinstance(data, (pd.DataFrame, pd.Series)):
        return data.values
    elif isinstance(data, np.ndarray):
        return data
    else:
        return np.array(data)

In [39]:
my_gbdt = MyGradientBoosting(max_depth=15, number_of_trees=5, max_features=10, base_learner=MyDecisionTreeRegressor, random_state=21, learning_rate=0.001)
my_gbdt.fit(train_encoded, Y_train)

Дерево 0, Loss 0.3556
Дерево 1, Loss 0.3556
Дерево 2, Loss 0.3555
Дерево 3, Loss 0.3554
Дерево 4, Loss 0.3554


In [40]:
Y_pred_my_proba = my_gbdt.predict_proba(valid_encoded)[:,1]
my_gini = gini_21(Y_valid, Y_pred_my_proba)
print(f"My gini_score: {my_gini:.4f}")

My gini_score: 0.3603


In [41]:
skl_gbdt = GradientBoostingClassifier(max_depth=15, n_estimators=5, max_features=10, random_state=21, learning_rate=0.001,
                                      min_samples_split=2, min_samples_leaf=1)
skl_gbdt.fit(train_encoded, Y_train)

GradientBoostingClassifier(learning_rate=0.001, max_depth=15, max_features=10,
                           n_estimators=5, random_state=21)

In [42]:
Y_pred_skl_proba = skl_gbdt.predict_proba(valid_encoded)[:,1]
skl_gini = gini_21(Y_valid, Y_pred_skl_proba)
print(f"Skl gini_score: {skl_gini:.4f}")
# 0.3345

Skl gini_score: 0.3630


* The implementation is slow, but the classification quality is comparable with the given parameters.



###  Use LightGBM, Catboost, and XGBoost for fitting on a training set and prediction on a validation set.

* Review the documentation of the libraries and fine-tune the algorithms for the task.
* Note key differences between each implementation.
* Analyze special features of each algorithm (how does "categorical feature"  * work in Catboost, what is DART mode in XGBoost)?
 Which GBDT model gives the best result? Can you explain why?

### LightGBM

Features:
Gradient-based One-Side Sampling (GOSS) - One-sided sampling based on gradients - keeps data samples with larger errors and selectively samples those with small errors for further training, which speeds up learning.

Exclusive Feature Bundling (EFB) - Exclusive Feature Bundling - combines sparse mutually exclusive features into one, which reduces dimensionality and again speeds up training.

Pros:
No preprocessing needed for categorical features

High speed

Scalable computations (can run on GPU)

Cons:
High risk of overfitting on small datasets

Poor interpretability

For LGBMClassifier

Distinctive Parameters:
boosting_type - which boosting type is used (gbdt - gradient boosting, rf - random forest, dart - Dropouts meet Multiple Additive Regression Trees - randomly drops some trained trees from subsequent predictions - prevents overfitting)

subsample_for_bin - number of samples for binning

objective - learning objective and corresponding loss function (regression, binary, multiclass)

subsample, subsample_freq, colsample_bytree - fraction of rows and features for training each tree

reg_alpha, reg_lambda - L1 and L2 regularization

importance_type - feature importance algorithm type (split - frequency of use, or gain - total gain)

linear_tree - changes algorithm behavior: instead of predicting constant values in leaves (like classic GBDT), builds simple linear regression in each leaf based on features used in splits leading to that leaf

Common Parameters:
num_leaves

max_depth

learning_rate

n_estimators

class_weight (for multiclass)

min_split_gain - minimum loss reduction for split

min_child_weight, min_child_samples - minimum amount of data in leaf

random_state

n_jobs

In [69]:
train_df_for_lgb = pd.read_csv("../datasets/training.csv")
train_df_for_lgb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   RefId                              72983 non-null  int64  
 1   IsBadBuy                           72983 non-null  int64  
 2   PurchDate                          72983 non-null  object 
 3   Auction                            72983 non-null  object 
 4   VehYear                            72983 non-null  int64  
 5   VehicleAge                         72983 non-null  int64  
 6   Make                               72983 non-null  object 
 7   Model                              72983 non-null  object 
 8   Trim                               70623 non-null  object 
 9   SubModel                           72975 non-null  object 
 10  Color                              72975 non-null  object 
 11  Transmission                       72974 non-null  obj

In [70]:
cat_cols = train_df_for_lgb.select_dtypes(include=['object']).columns
date_cols = ['PurchDate']
cat_cols = [col for col in cat_cols if col not in date_cols]
train_df_for_lgb[cat_cols] = train_df_for_lgb[cat_cols].fillna('MISSING')

In [71]:
for column in cat_cols:
    train_df_for_lgb[column] = train_df_for_lgb[column].astype('category')

In [74]:
def tvt_split_by_time(train_df:pd.DataFrame):
  total = len(train_df)
  train_end_idx = int(total * 0.33)
  val_end_idx = int(total * 0.66)

  train_df["PurchDate"] = pd.to_datetime(train_df["PurchDate"], errors='coerce')
  train_df = train_df.sort_values(by="PurchDate")
  train_df["PurchDate"] = train_df["PurchDate"].astype(int)

  X = train_df.drop(["IsBadBuy"], axis=1)
  Y = train_df["IsBadBuy"]

  while X.loc[train_end_idx]["PurchDate"] == X.loc[val_end_idx]["PurchDate"]:
    val_end_idx +=1
    print(val_end_idx)

  X_train = X.iloc[:train_end_idx]
  X_valid = X.iloc[train_end_idx:val_end_idx]
  X_test = X.iloc[val_end_idx:]

  Y_train = Y.iloc[:train_end_idx]
  Y_valid = Y.iloc[train_end_idx:val_end_idx]
  Y_test = Y.iloc[val_end_idx:]
  return X_train, X_valid, X_test, Y_train, Y_valid, Y_test

In [75]:
X_train_lgb, X_valid_lgb, X_test_lgb, Y_train_lgb, Y_valid_lgb, Y_test_lgb = tvt_split_by_time(train_df_for_lgb)


In [76]:
lgb_model = lgb.LGBMClassifier(max_depth=15, learning_rate=0.1, n_estimators=100, random_state=21, linear_tree=True)
lgb_model.fit(X_train_lgb, Y_train_lgb)
     

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 2756, number of negative: 21328
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002086 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4487
[LightGBM] [Info] Number of data points in the train set: 24084, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.114433 -> initscore=-2.046240
[LightGBM] [Info] Start training from score -2.046240


LGBMClassifier(linear_tree=True, max_depth=15, random_state=21)

### catboost

Features:
Ordered boosting - each subsequent model is trained on errors of the previous model using "honest" gradient (errors on data that the previous model did not see during training). Prevents overfitting.

Efficient handling of categorical features - no need for preprocessing into numeric representation.

Pros:
No preprocessing needed for categorical features

Overfitting prevention built-in

Handles NaN values (but note: processes them automatically for numerical features, requires explicit replacement of strings with np.nan for categorical features)

Interpretability (built-in tools for feature importance, SHAP)

Cons:
Relatively long training time

Increased memory consumption

Main parameters for CatBoostClassifier:
Distinctive parameters:
cat_features - indices of categorical columns

l2_leaf_reg, random_strength, use_best_model, early_stopping_rounds - for regularization

bootstrap_type - type of bootstrap (Bayesian, Bernoulli, MVS, Poisson)

subsample - fraction of samples for training each tree

grow_policy - tree growth strategy (SymmetricTree - fast, default; Depthwise; Lossguide)

one_hot_max_size - threshold for one-hot encoding of categorical features

max_ctr_complexity - maximum complexity of categorical feature combinations for creating new features

task_type - GPU or CPU

boosting_type - boosting type ('Ordered' - default, prevents overfitting; or 'Plain' - classic gradient boosting)

Common parameters:
iterations / n_estimators (mutually exclusive)

depth

learning_rate

loss_function

max_leaves, min_data_in_leaf

random_state

In [79]:
train_df_for_cb = pd.read_csv("../datasets/training.csv")
train_df_for_cb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   RefId                              72983 non-null  int64  
 1   IsBadBuy                           72983 non-null  int64  
 2   PurchDate                          72983 non-null  object 
 3   Auction                            72983 non-null  object 
 4   VehYear                            72983 non-null  int64  
 5   VehicleAge                         72983 non-null  int64  
 6   Make                               72983 non-null  object 
 7   Model                              72983 non-null  object 
 8   Trim                               70623 non-null  object 
 9   SubModel                           72975 non-null  object 
 10  Color                              72975 non-null  object 
 11  Transmission                       72974 non-null  obj

In [80]:
cat_cols = train_df_for_cb.select_dtypes(include=['object']).columns
train_df_for_cb[cat_cols] = train_df_for_cb[cat_cols].fillna('MISSING')
X_train_cb, X_valid_cb, X_test_cb, Y_train_cb, Y_valid_cb, Y_test_cb = tvt_split_by_time(train_df_for_cb)

In [81]:
categorical_feature_indices = [i for i, cat in enumerate(X_train_cb.columns) if X_train_cb[cat].dtype == 'object']


In [85]:
catboost_model = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, verbose=0)
catboost_model.fit(X_train_cb, Y_train_cb, cat_features=categorical_feature_indices)
     

CatBoostClassifier(depth=6, iterations=500, learning_rate=0.1, verbose=0)

In [86]:
y_pred = catboost_model.predict_proba(X_valid_cb)[:,1]
catboost_gini = gini_21(Y_valid_cb, y_pred)
print(f"Catboost gini_score: {catboost_gini:.4f}")
# 0.4671

Catboost gini_score: 0.4664


### XGBoost = eXtreme Gradient Boosting

Features:
As of version 1.5, no preprocessing needed for categorical features

Several built-in techniques to prevent overfitting: learning rate, regularization, tree pruning

Tree construction: breadth-first (not depth-first)

Handles NaN values

Caching

Uses approximate calculations instead of greedy algorithm for finding the best feature splits

Pros:
Good for large datasets

Support for parallelization and GPU computations

Customizable parameters and regularization

Built-in tools for feature importance

Cons:
Computationally expensive

Sensitive to noise and outliers

Limited interpretability

Main parameters for XGBClassifier:
Distinctive parameters:
enable_categorical - support for categorical features

booster - model type (gbtree - default, dart - with dropout, gblinear)

tree_method - tree construction algorithm (auto, exact, approx, hist, gpu_hist)

importance_type - feature importance calculation algorithm (weight - frequency, gain, cover, total_gain, total_cover)

subsample, colsample_bytree, colsample_bylevel, colsample_bynode - fraction of samples and features for building each tree, fraction of features for each level and node

scale_pos_weight - weight for positive class (important for imbalance)

reg_lambda, reg_alpha - L2, L1 regularization

gamma (min_split_loss) - minimum loss reduction for split

device - GPU or CPU

Common parameters:
n_estimators

max_depth

learning_rate (eta)

objective

random_state

In [87]:
train_df_for_xgb = pd.read_csv("../datasets/training.csv")
train_df_for_xgb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   RefId                              72983 non-null  int64  
 1   IsBadBuy                           72983 non-null  int64  
 2   PurchDate                          72983 non-null  object 
 3   Auction                            72983 non-null  object 
 4   VehYear                            72983 non-null  int64  
 5   VehicleAge                         72983 non-null  int64  
 6   Make                               72983 non-null  object 
 7   Model                              72983 non-null  object 
 8   Trim                               70623 non-null  object 
 9   SubModel                           72975 non-null  object 
 10  Color                              72975 non-null  object 
 11  Transmission                       72974 non-null  obj

In [88]:
cat_cols = train_df_for_xgb.select_dtypes(include=['object']).columns
date_cols = ['PurchDate']
cat_cols = [col for col in cat_cols if col not in date_cols]
train_df_for_xgb[cat_cols] = train_df_for_xgb[cat_cols].fillna('MISSING')

In [89]:
for column in cat_cols:
    train_df_for_xgb[column] = train_df_for_xgb[column].astype('category')


In [91]:
X_train_xgb, X_valid_xgb, X_test_xgb, Y_train_xgb, Y_valid_xgb, Y_test_xgb = tvt_split_by_time(train_df_for_xgb)


In [92]:
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    enable_categorical=True,
    n_estimators=50,
    max_depth=6,
    learning_rate=0.1,
    random_state=21
)
xgb_model.fit(X_train_xgb, Y_train_xgb)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=50,
              n_jobs=None, num_parallel_tree=None, ...)

In [93]:
_pred = xgb_model.predict_proba(X_valid_xgb)[:,1]
xgb_gini = gini_21(Y_valid_cb, y_pred)
print(f"Xgb gini_score: {xgb_gini:.4f}")
# 0.4513
     

Xgb gini_score: 0.4664


* Overall, all three models are comparable, but CatBoost showed slightly better results - serving as the benchmark for working with datasets that have many categorical features.


#### Take the best model and estimate its performance on the test dataset:

* check the Gini values on all three datasets for your best model: training Gini, valid Gini, test Gini.
* Do you see a drop in performance when comparing the valid quality to the test quality?
* Is your model overfitting or not? Explain.

In [94]:
y_pred = catboost_model.predict_proba(X_train_cb)[:,1]
catboost_gini = gini_21(Y_train_cb, y_pred)
print(f"Catboost train valid gini_score: {catboost_gini:.4f}")
     

Catboost train valid gini_score: 0.7397


In [95]:
y_pred = catboost_model.predict_proba(X_valid_cb)[:,1]
catboost_gini = gini_21(Y_valid_cb, y_pred)
print(f"Catboost valid gini_score: {catboost_gini:.4f}")

Catboost valid gini_score: 0.4664


In [96]:
y_pred = catboost_model.predict_proba(X_test_cb)[:,1]
catboost_gini = gini_21(Y_test_cb, y_pred)
print(f"Catboost test gini_score: {catboost_gini:.4f}")

Catboost test gini_score: 0.4375


#### Overfitting exists because the metric quality on validation and test sets is about 1.5 times lower. You can try to improve the situation using built-in mechanisms

In [97]:
catboost_model = CatBoostClassifier(
    iterations=1000,
    use_best_model=True,
    od_type='Iter',
    od_wait=20,
    eval_metric='AUC',
    learning_rate=0.1,
    depth=6,
    verbose=0)
catboost_model.fit(X_train_cb, Y_train_cb,
                   cat_features=categorical_feature_indices,
                   eval_set=(X_valid_cb, Y_valid_cb),
                   verbose=50)
     

0:	test: 0.6929583	best: 0.6929583 (0)	total: 95.1ms	remaining: 1m 35s
50:	test: 0.7431260	best: 0.7431271 (49)	total: 3.78s	remaining: 1m 10s
Stopped by overfitting detector  (20 iterations wait)

bestTest = 0.7433531497
bestIteration = 73

Shrink model to first 74 iterations.


CatBoostClassifier(depth=6, eval_metric='AUC', iterations=1000, learning_rate=0.1, od_type='Iter', od_wait=20, use_best_model=True, verbose=0)

In [107]:
y_pred = catboost_model.predict_proba(X_train_cb)[:,1]
catboost_gini = gini_21(Y_train_cb, y_pred)
print(f"Catboost train valid gini_score: {catboost_gini:.4f}")

Catboost train valid gini_score: 0.5682


In [106]:
y_pred = catboost_model.predict_proba(X_valid_cb)[:,1]
catboost_gini = gini_21(Y_valid_cb, y_pred)
print(f"Catboost valid gini_score: {catboost_gini:.4f}")

Catboost valid gini_score: 0.4867


In [105]:
y_pred = catboost_model.predict_proba(X_train_cb)[:,1]
catboost_gini = gini_21(Y_train_cb, y_pred)
print(f"Catboost train valid gini_score: {catboost_gini:.4f}")

Catboost train valid gini_score: 0.5682


###  Implement the ExtraTreesClassifier and check its performance. You must improve the result of a single tree and obtain a Gini score of at least 0.12 on the validation dataset.

In [99]:
class MyExtraTreesClassifier(MyRandomForestClassifier):
    def __init__(self, n_estimators=100, criterion='gini', max_depth=15,
                 min_samples_split=100, min_samples_leaf=1, n_features=10,
                 random_state=21, max_samples=None, n_random_thresholds=5):
        super().__init__(n_estimators=n_estimators,
                        criterion=criterion,
                        max_depth=max_depth,
                        min_samples_split=min_samples_split,
                        min_samples_leaf=min_samples_leaf,
                        n_features=n_features,
                        random_state=random_state,
                        max_samples=max_samples)
        self.n_random_thresholds = n_random_thresholds

    def fit(self, X, Y):
        X = self._to_numpy(X)
        Y = self._to_numpy(Y)
        self.classes_ = np.unique(Y)
        self.n_classes_ = len(self.classes_)
        self.trees = []
        rng = np.random.default_rng(self.random_state)

        for i in range(self.n_estimators):
            tree_seed = rng.integers(0, 2**31 - 1)
            X_sample, Y_sample = self._bootstrap_sample(X, Y, rng)

            tree = MyExtraRandomizedTree(
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                max_depth=self.max_depth,
                n_features=self.n_features,
                criterion=self.criterion,
                random_state=tree_seed,
                n_random_thresholds=self.n_random_thresholds
            )

            tree.fit(X_sample, Y_sample)
            self.trees.append(tree)

In [100]:
my_classifier = MyExtraTreesClassifier(n_estimators=50,max_depth=15, min_samples_split=10, n_features=10, max_samples=100, n_random_thresholds=5)
my_classifier.fit(train_encoded, Y_train)
     

In [101]:
Y_pred_my = my_classifier.predict_proba(valid_encoded)[:,1]
my_gini = gini_21(Y_valid, Y_pred_my)
print(f"My gini_score: {my_gini:.4f}")
# 0.23 - 0.25

My gini_score: 0.2469


In [104]:
skl_classifier = ExtraTreesClassifier(n_estimators=50,max_depth=15, min_samples_split=10, max_features=10, bootstrap=True, max_samples=100)
skl_classifier.fit(train_encoded, Y_train)

Y_pred_skl = skl_classifier.predict_proba(valid_encoded)[:,1]
skl_gini = gini_21(Y_valid, Y_pred_skl)
print(f"Skl gini_score: {skl_gini:.4f}")
# 0.31 - 0.39
     

Skl gini_score: 0.3387
